In [0]:
%run /Workspace/Users/vamshialwayskush@gmail.com/Brazilian_ECommerce/ImportPackages

In [0]:
from datetime import timedelta, datetime
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window as W
from pyspark.storagelevel import StorageLevel
from delta.tables import *
from functools import reduce

import os #for adding filename to dataframe
import re #regular expressions
import time

In [0]:
dbutils.widgets.text("sourcefiles_path", "/Volumes/mde_dev/volumes/aw_source/")
dbutils.widgets.text("delta_schema", "bronze")
dbutils.widgets.text("catalog_name", "mde_dev")

pSolumeFilesPath = dbutils.widgets.get("sourcefiles_path")
pDeltaSchema = dbutils.widgets.get("delta_schema")
pCatalogName = dbutils.widgets.get("catalog_name")

print("pSolumeFilesPath: " + pSolumeFilesPath)
print("pCatalogName: " + pCatalogName)
print("pDeltaSchema: " + pDeltaSchema)


In [0]:
%sql
CREATE DATABASE IF NOT EXISTS mde_dev.gold;
CREATE DATABASE IF NOT EXISTS mde_dev.silver;

In [0]:
%python
try:
    print(f"Checking if {pCatalogName}.{pDeltaSchema} schema exists or not")
    schema_exists = spark.sql(f" DESCRIBE DATABASE {pCatalogName}.{pDeltaSchema}")
    success = True
    if schema_exists:
        print(f"{pCatalogName}.{pDeltaSchema} already exist")
except Exception as e:
    create_if_not_exist = True
    if create_if_not_exist:
        try:
            print("creating schema as schema does not exists")
            spark.sql(f"CREATE SCHEMA IF NOT EXISTS {pCatalogName}.{pDeltaSchema}")
            print(f"{pCatalogName}.{pDeltaSchema} schema created")
            success = True
        except Exception as e:
            print(f"Error creating schema: {e}")
            success = False


In [0]:
if success:
    try:
        # List all files in the DBFS folder
        files = dbutils.fs.ls(f"{pSolumeFilesPath}")
        # print(files)

        # Filter for CSV files and extract only the file names
        csv_file_names = [f.name for f in files if f.name.endswith(".csv")]

        # Display the file names
        print(csv_file_names)

        success = True
    except Exception as e:
        print(f"Error listing files: {e}")
        success = False

In [0]:
success = True

csv_file_names = [ 'SalesOrderDetail.csv', 'SalesOrderHeader.csv', 'SalesOrderHeaderSalesReason.csv', 'SalesPerson.csv', 'SalesReason.csv', 'ShipMethod.csv', 'SpecialOffer.csv', 'SpecialOfferProduct.csv', 'StateProvince.csv', 'Store.csv']

if success:
    try:
        for file in csv_file_names:
            file_name = file

            # file path and file name
            file_path_name = f"{pSolumeFilesPath}{file_name}"
            # raw table name
            delta_table_name = file_name.replace(".csv", "").strip()

            if delta_table_name.endswith("_dataset"):
                delta_table_name = delta_table_name.replace("_dataset", "")
            else:
                delta_table_name

            print("file path and name: " + file_path_name)
            print("delta table name: " + delta_table_name)

            target_file_location_name = f"{pCatalogName}.{pDeltaSchema}.{delta_table_name}"
            
            print(f"Target table name: {target_file_location_name}")
            
            print(f"----- Reading {file_name} from {pSolumeFilesPath} -----")

            df = spark.read.format("csv")\
                .option("mode", "FAILFAST")\
                .option("inferSchema", True)\
                .option("header", True)\
                .load(f"{file_path_name}")
                
            # Drop Duplicates
            df = df.dropDuplicates()

            print(f"----- Processing {delta_table_name} into {target_file_location_name} -----")
            df.write.format("delta").mode("append").saveAsTable(f"{target_file_location_name}")
            print(f"----- Processed {delta_table_name} into {target_file_location_name} -----")

    except Exception as e:
        print(f"Error: {e}")
        success = False


In [0]:
# success = True

# csv_file_names = ['olist_customers_dataset.csv']

# if success:
#     try:
#         for file in csv_file_names:
#             file_name = file
#             # file path and file name
#             file_path_name = f"{file_path}{file_name}"
#             # raw table name
#             target_table_name = file_name.replace(".csv", "")

#             if target_table_name.endswith("_dataset"):
#                 target_table_name = target_table_name.replace("_dataset", "")
#             else:
#                 target_table_name
            
#             # Create a view to use in SQL statements      
#             temp_vw_name_raw = 'tmpvw_raw_' + target_table_name

#             print("file path and name: " + file_path_name)
#             print("temporary view name: " + temp_vw_name_raw)
#             print("target table name: " + target_table_name)

#             target_file_location_name = f"{catalog_name}.{target_schema}.{target_table_name}"
#             print(f"Target table name: {target_file_location_name}")
            
#             print(f"----- Reading {file_name} from {file_path} -----")

#             df = spark.read.format("csv")\
#                 .option("inferSchema", True)\
#                 .option("header", True)\
#                 .load(f"{file_path_name}")

#             #  .option("mode", "FAILFAST")\
                
#             # Drop Duplicates
#             df = df.dropDuplicates()

#             df.createOrReplaceTempView(temp_vw_name_raw)

#             print(f"----- Creating temporary view {temp_vw_name_raw} -----")
#             sdf = spark.table(temp_vw_name_raw) # 11s 

#             columns = [f"`{column}`" for column in sdf.columns]

#             print(f"----- Adding metadata columns to temporary view {temp_vw_name_raw} -----")

#             # FIRST: Add sha2 hash over all source columns for SCD Type 2
#             sdf = sdf.withColumn("dl_rowhash", sha2(concat_ws("||", *columns), 256))
                
#             # Add an Active flag, setting it to True
#             sdf = sdf.withColumn("dl_iscurrent", lit(True)) #Add column indicating the pDeltaObjectName #2.24m

#             # Add an IsDeleted flag, setting it to False
#             sdf = sdf.withColumn("dl_isdeleted", lit(False))
            
#             # Add Record dates for SCD Type 2
#             sdf = sdf.withColumn('dl_recordstartdateutc', current_timestamp()) #add timestamp when processed
#             sdf = sdf.withColumn("dl_recordenddateutc", to_timestamp(lit(None), "yyyy-MM-dd HH:mm:ss")) #Add column indicating the type 2 end date

#             # Create a temp table from Dataframe to use in MERGE statement into Delta Lake
#             vw_name_enriched = 'tmpvw_enriched_' + temp_vw_name_raw

#             sdf.createOrReplaceTempView(vw_name_enriched)

#             sdf1 = spark.table(vw_name_enriched)
#             print(f"----- Created temporary view {vw_name_enriched} -----")

#             # print(f"----- Processing {file_name} into {target_file_location_name} -----")
#             # sdf1.write.format("delta").mode("append").saveAsTable(f"{target_file_location_name}")
#             # print(f"----- Processed {file_name} into {target_file_location_name} -----")

#             if success:
#                 try:
#                     table_exists =  spark.sql(f"DESCRIBE TABLE {catalog_name}.{target_schema}.{target_table_name}")
#                     success = True
#                 except Exception:
#                     sucess = False

#             if table_exists and success:
#                 try:
#                     print(f"{datetime.now()}: Table exists. Started full load merge of new dataset into into base table...")

#                     delta_path = target_file_location_name+"/_delta_log"
#                     target_delta = DeltaTable.forPath(spark, delta_path)
#                     target_df = target_delta.toDF()

#                     ############################################################################
#                     # 1. Update/expire current rows in target table.  These are
#                     #    rows where there is no match on row hash in incoming dataset.
#                     #    Since this is a FULL (not incremental) dataset, 
#                     #    Update all rows in table where the rowhash is not in the new dataset
#                     ############################################################################

#                     print(f"{datetime.now()}: Updating/expiring current rows in base table...")

#                     # Create a dataframe of row hashes in table that are not in new file 
#                     # Note:  Only need records where they are not already inactive.
#                     updateDF = (target_df.alias("target")
#                         .join(sdf.select("dl_rowhash").alias("source"),
#                             (col("target.dl_iscurrent") == "True") 
#                                 & (col("source.dl_rowhash") == col("target.dl_rowhash")
#                             ), 
#                             "left_outer"
#                             ).filter("source.dl_rowhash IS NULL AND target.dl_iscurrent == 'True'"))

#                     # UPDATE (EXPIRE) all records in table that do not have a corresponding row hash in new file
#                     target_delta.alias("target").merge(
#                         source = updateDF.select("target.*").alias("expired"), 
#                         condition = expr(f"""target.dl_iscurrent = true  
#                             AND expired.dl_rowhash = target.dl_rowhash"""
#                         )
#                     ).whenMatchedUpdate(
#                         set = {
#                             "target.dl_iscurrent": "false",
#                             "target.dl_recordenddateutc": f"'{BATCH_TIME_STR}'",
#                             "target.dl_isdeleted": "true"
#                         }
#                     ).execute()


#                     ############################################################################
#                     # 2. Insert new records from updates dataset where there 
#                     #    is no match on the row hash (and is already current)
#                     ############################################################################
#                     print(f"{datetime.now()}: Adding new (or updated) rows to base table...")

#                     # FOR SCHEMA EVOLUTION:
#                     #   Create a list of columns to use in MERGE statement
#                     #   that has all columns from the base table and incoming dataset
#                     diffcols = list(set(sdf.columns)-set(target_df.columns)) # Diff incoming dataset with basetable
#                     added_cols = target_df.columns + diffcols # Add any new columns from dataset to basetable columns
#                     diffcols = list(set(target_df.columns)-set(sdf.columns)) # Diff incoming dataset with target
#                     all_columns = ["source." + col for col in added_cols if not col in diffcols ] # all columns, minus any dropped columns, aliased for MERGE

#                     # Refresh dataframe (due to previous merge)
#                     target_df = target_delta.toDF()
                    
#                     # Create a dataframe of records in new file that do not have
#                     # a corresponding row hash already in table, that is active
#                     insertDF = (sdf.alias("source")
#                         .join(target_df.alias("target"),
#                             (col("target.dl_iscurrent") == "True") #MUST HAVE THIS INCLUDED FOR THIS TO WORK PROPERLY
#                             &  (col("source.dl_rowhash") == col("target.dl_rowhash")) #Note, there can be existing records with this hash
#                             , "left_outer"
#                         ).filter("target.dl_rowhash IS NULL"))
                    

#                     # INSERT new records into table, using row hash for key
#                     target_delta.alias("target").merge(
#                         #source = sdf.select(all_columns).alias("updates"), 
#                         source = insertDF.select(all_columns).alias("source"), 
#                         condition = expr(f"""source.dl_rowhash == target.dl_rowhash 
#                             AND target.dl_iscurrent = true"""
#                         )
#                     ).whenNotMatchedInsertAll(
#                     ).execute()

#                     print(f"{datetime.now()}: Completed full load merge of new dataset into into base table...")
#                 except Exception as E:
#                     output_message = f"ERROR merging full file! {str(E)[:100]}"
#                     print(output_message)
#                     success = False
#                 except:
#                     output_message = f"ERROR merging full file!"
#                     print(output_message)
#                     success = False
                
#                 if success:
#                     print(f"----- Processing {file_name} into {target_file_location_name} -----")
                    
#                     sdf1.write.format("delta").mode("append").saveAsTable(f"{target_file_location_name}")
                    
#                     print(f"----- Processed {file_name} into {target_file_location_name} -----")
    

#     except Exception as e:
#         print(f"Error: {e}")
#         success = False
